# 2. LLM 활용 (Gemma)

In [3]:
!pip install --upgrade pip

  Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 22.3
    Uninstalling pip-22.3:
      Successfully uninstalled pip-22.3


In [20]:
!pip install triton
!pip install transformers==4.41.2
!pip install accelerate==0.28.0 ### 0.31.0에서 LoRa때문에 버전 내림
!pip install -U bitsandbytes
!pip install peft==0.10.0
!pip install datasets

python(68547) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement triton (from versions: none)
ERROR: No matching distribution found for triton


python(68554) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


python(68556) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


python(68558) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


python(68559) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


python(68561) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 1) Import 

In [ ]:
import pandas as pd 
import torch 
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
import os

# 디바이스 설정
if torch.backends.mps.is_available():
    device = "mps"
    use_quantization = False  # MPS에서는 양자화 미지원
    print("Using MPS (Metal Performance Shaders)")
elif torch.cuda.is_available():
    device = "cuda"
    use_quantization = True
    print("Using CUDA")
else:
    device = "cpu"
    use_quantization = False
    print("Using CPU")

Using MPS (Metal Performance Shaders)


## 2) Data Load

In [22]:
train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

In [23]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

# 학습용 데이터셋 준비
def format_prompt(input_text, output_text):
    system_prompt = (
        "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
        "Your task is to transform the given obfuscated Korean review into a clear, correct, "
        "and natural-sounding Korean review that reflects its original meaning. "
        "Spacing and word length in the output must be restored to the same as in the input. "
        "Do not provide any description. Print only in Korean.\n\n"
    )
    prompt = f"{system_prompt}input : {input_text}\noutput : {output_text}"
    return prompt

train_texts = []
for i in range(len(train)):
    prompt = format_prompt(train['input'][i], train['output'][i])
    train_texts.append(prompt)

train_dataset = Dataset.from_dict({"text": train_texts})

## 3) Model Load

In [ ]:
model_id = 'beomi/gemma-ko-2b'

# 양자화 설정 (CUDA에서만 사용)
if use_quantization:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        quantization_config=bnb_config, 
        device_map={"": 0}
    )
else:
    # MPS나 CPU에서는 양자화 없이 로드
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "mps" else torch.float32,
        device_map=device if device != "cpu" else None
    )
    if device == "cpu":
        model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# LoRA 설정
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA alpha
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Gemma의 attention 모듈
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# 모델을 LoRA 학습 준비
if use_quantization:
    model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


  0%|          | 0/2112 [01:34<?, ?it/s]


Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.72s/it]


trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.14687678751090355
Trainable: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight


## 4) LoRA Fine-tuning


In [ ]:
# 데이터셋 토크나이징
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 학습 설정
training_args = TrainingArguments(
    output_dir="./lora_gemma_output",
    num_train_epochs=3,
    per_device_train_batch_size=4 if use_quantization else 2,  # 양자화 사용 시 더 큰 배치
    gradient_accumulation_steps=4 if use_quantization else 8,
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    learning_rate=2e-4,
    fp16=(device == "mps" or use_quantization),  # MPS나 양자화 사용 시 fp16
    optim="paged_adamw_8bit" if use_quantization else "adamw_torch",
    save_total_limit=2,
    report_to="none",
    dataloader_pin_memory=False if device == "mps" else True,
)

# Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Trainer 설정
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 학습 실행
trainer.train()

# 모델 저장
model.save_pretrained("./lora_gemma_model")
tokenizer.save_pretrained("./lora_gemma_model")


Map: 100%|██████████| 11263/11263 [00:01<00:00, 8056.69 examples/s]
/Users/itaewon/.pyenv/versions/3.11.0/lib/python3.11/site-packages/accelerate/accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/Users/itaewon/.pyenv/versions/3.11.0/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


KeyboardInterrupt: 

## 5) Inference with Fine-tuned Model


In [ ]:
# Fine-tuned 모델 로드 (학습 후 사용)
from peft import PeftModel

# Base 모델 다시 로드
if use_quantization:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        quantization_config=bnb_config, 
        device_map={"": 0}
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "mps" else torch.float32,
        device_map=device if device != "cpu" else None
    )
    if device == "cpu":
        base_model = base_model.to(device)

# LoRA 가중치 로드
model = PeftModel.from_pretrained(base_model, "./lora_gemma_model")
model = model.merge_and_unload()  # LoRA 가중치를 base 모델에 병합

pipe = pipeline(
    task="text-generation",
    model=model, 
    tokenizer=tokenizer 
)

restored_reviews = []

for index, row in test.iterrows():
    query = row['input']  
    
    # Fine-tuned 모델은 학습 시 사용한 포맷과 동일하게
    system_prompt = (
        "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
        "Your task is to transform the given obfuscated Korean review into a clear, correct, "
        "and natural-sounding Korean review that reflects its original meaning. "
        "Spacing and word length in the output must be restored to the same as in the input. "
        "Do not provide any description. Print only in Korean.\n\n"
    )
    prompt = f"{system_prompt}input : {query}\noutput : "

    outputs = pipe(
        prompt,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=len(query) + 50,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    
    generated_text = outputs[0]['generated_text']
    result = generated_text[len(prompt):].strip()
    
    # "output : " 이후의 텍스트만 추출
    if "output :" in result:
        result = result.split("output :")[-1].strip()
    
    restored_reviews.append(result)

## 6) Submission

In [23]:
submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = restored_reviews

In [ ]:
submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')